In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def load_heights(/content/GaltonFamilies.csv, parent="father"):
  df= pd.read_csv(/content/GaltonFamilies.csv)
  parents= df.drop_duplicates("family")[parent].to_numpy(float)
  children= df["childHeight"].to_numpy(float)
  return np.concatenate([parents, children]), parents, children

In [3]:
def gaussian_pdf(x, mu, var):
  return np.exp(-((x - mu) **2) / (2 * var)) / np.sqrt(2 * np.pi * var)

In [4]:
def log_likelihood(x, mu, var, pi):
  mix = sum(pi[k] * gaussian_pdf(x, mu[k], var[k]) for k in range (len(mu)))
  return float(np.sum(np.log(mix)))

In [5]:
def e_step(x, mu, var, pi):
  weighted = np.column_stack([pi[k] * gaussian_pdf(x, mu[k], var[k]) for k in range (len(mu))])
  return weighted / weighted.sum(axis=1, keepdims=True)

In [6]:
def m_step(x, r):
  nk = r.sum(axis=0)
  mu = (r * x[:, None]).sum(axis=0) / nk
  var = (r * (x[:, None] - mu) **2).sum(axis=0) / nk
  pi = nk / len(x)
  return mu, var, pi

In [7]:
def init_params(x, children_is_taller):
  lo, hi = np.percentile(x, [25, 75])
  mu = np.array([hi, lo]) if children_is_taller else np.array([lo, hi])
  return mu, np.array([x.var(), x.var()]), np.array([0.5, 0.5])

In [8]:
def fit_em(x, children_is_taller, tol=1e-6, max_iter = 500):
  mu, var, pi = init_params(x, children_is_taller)
  ll = log_likelihood(x, mu, var, pi)
  history = [(0, *mu, *var,  *pi, ll)]
  for i in range(1, max_iter + 1):
    r = e_step(x, mu, var, pi)
    mu, var, pi = m_step(x, r)
    new_ll = log_likelihood(x, mu, var, pi)
    history.append((i, *mu, *var, *pi, new_ll))
    if np.abs(new_ll - ll) < tol:
      break
    ll = new_ll
  return mu, var, pi, history

In [9]:
def tracking_table(history, rows=(0, 1, 2)):
  cols = ["Iteration", "mu1 (Children)", "mu2 (Fathers)", "var1", "var2", "pi1", "pi2", "log-likelihood"]
  df = pd.DataFrame(history, columns=cols)
  return df[df["Iteration"].isin(rows)].round(4).reset_index(drop=True)

In [10]:
def classify(height, mu, var, pi, names=("Child", "Father")):
  w = np.array([pi[k] * gaussian_pdf(height, mu[k], var[k]) for k in range (len(mu))])
  post = w / w.sum()
  return dict(zip(names, (float(p) for p in post)))

In [11]:
def global_mean_split(x, truth_is_child):
  cut = x.mean()
  pred_high = x > cut
  return cut, float(np.mean(pred_high == truth_is_child))

In [12]:
def plot_fit(x, mu, var, pi, path=None, names=("Children", "Fathers")):
    grid = np.linspace(x.min() - 2, x.max() + 2, 400)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(x, bins=30, density=True, alpha=0.35, color="gray", label="Pooled heights")
    for k, c in zip(range(2), ("tab:blue", "tab:red")):
        ax.plot(grid, pi[k] * gaussian_pdf(grid, mu[k], var[k]), color=c, label=f"{names[k]} component")
    ax.plot(grid, sum(pi[k] * gaussian_pdf(grid, mu[k], var[k]) for k in range(len(mu))), "k--", label="Mixture")
    ax.axvline(x.mean(), color="green", ls=":", label="Global mean")
    ax.set_xlabel("Height (inches)"); ax.set_ylabel("Density"); ax.legend(); ax.set_title("EM fit: Children vs Fathers")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

def plot_loglik(history):
    plt.figure(figsize=(7, 4))
    plt.plot([h[0] for h in history], [h[-1] for h in history])
    plt.xlabel("Iteration"); plt.ylabel("Log-likelihood"); plt.title("Convergence")
    plt.tight_layout(); plt.show()

In [15]:
parent = "father"
test_h = 66.0

x, parents, children = load_heights("GaltonFamilies.csv", "father")
taller = children.mean() > parents.mean()
print("initial params:", init_params(x, taller))
mu, var, pi, hist = fit_em(x, children_is_taller=taller)

pd.set_option("display.width", 200)
print(tracking_table(hist).to_string(index=False))
print(f"Converged after {hist[-1][0]} iterations, final params: mu={mu.round(3)}, var={var.round(3)}, pi={pi.round(3)}")
print(f"P(height={test_h}):", {k: round(v, 4) for k, v in classify(test_h, mu, var, pi).items()})

truth = np.concatenate([np.zeros(len(parents), bool), np.ones(len(children), bool)])
if not taller:
    truth = ~truth
cut, acc = global_mean_split(x, truth)
print(f"Global-mean split at {cut:.2f}: accuracy {acc:.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'GaltonFamilies.csv'

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(children, bins=30, alpha=0.5, density=True, label="Children", color="tab:blue")
plt.hist(parents, bins=20, alpha=0.5, density=True, label="Fathers", color="tab:red")
plt.xlabel("Height (inches)"); plt.ylabel("Density"); plt.legend()
plt.title("Raw heights: Children vs Fathers")
plt.show()

In [ ]:
plot_fit(x, mu, var, pi, path="em_fit.png")
plot_loglik(hist)